In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, MinMaxScaler
from sklearn.model_selection import train_test_split

# 1. Load data with 32-bit types to save memory
dtypes = {
    'user_id': 'int32',
    'video_id': 'int32',
    'play_duration': 'float32',
    'video_duration': 'float32',
    'watch_ratio': 'float32'
}
cols = ['user_id', 'video_id', 'video_duration', 'watch_ratio']
df = pd.read_csv('../data/raw/small_matrix.csv', usecols=cols, dtype=dtypes)
items = pd.read_csv('../data/raw/item_categories.csv')

# 2. Add video features
df = df.merge(items[['video_id', 'feat']], on='video_id', how='left')

# 3. Define CTR label (1 = completed/satisfied view)
df['label'] = (df['watch_ratio'] >= 1.0).astype(int)

# Rapid prototype sample (200k rows)
df_sample = df.sample(n=200000, random_state=42).reset_index(drop=True)

# 4. Encodings
for col in ['user_id', 'video_id', 'feat']:
    df_sample[col] = LabelEncoder().fit_transform(df_sample[col].astype(str))

mms = MinMaxScaler()
df_sample['video_duration'] = mms.fit_transform(df_sample[['video_duration']].fillna(0))

# 5. Split and save to Parquet
train_df, test_df = train_test_split(df_sample, test_size=0.2, random_state=42)
train_df.to_parquet('../data/processed/train.parquet', index=False)
test_df.to_parquet('../data/processed/test.parquet', index=False)
print("Data saved successfully!")

Data saved successfully!
